In [26]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "Artificial is intelligence is transforming the way people the interact with technology"
tokens = u_tokenizer.encode(prompt)

tokens

tensor([ 1, 63,  3, 63,  2, 63,  3, 63,  4, 63,  5, 63,  6, 63,  7, 63,  5, 63,
         8, 63,  9, 63, 10])

In [27]:
context_length = 32

In [28]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab),embedding_dim = 4,num_heads = 2,context_length= 32,num_layers = 3)

out = u_model(tokens)
out.shape

torch.Size([23, 65])

In [29]:
with open("text.txt", "r") as f:
    text = f.read()

len(text) ,text[:100]

(1970,
 'Artificial intelligence is transforming the way people interact with technology. Large language mode')

In [30]:
token_ids = u_tokenizer.encode(text)
len(token_ids) , type(token_ids)

(1000, torch.Tensor)

In [31]:
ids = token_ids.detach().cpu().numpy().tolist()
len(ids) ,type(ids)

(1000, list)

In [32]:
from text_dataset import TextDataset

stride = 2

dataset = TextDataset(ids , context_length,stride)

len(dataset.inputs) , len(dataset.targets)

(484, 484)

In [33]:
dataset.inputs[0],dataset.targets[0]

(tensor([ 1, 63,  2, 63,  3, 63,  4, 63,  5, 63,  6, 63,  7, 63,  8, 63,  9, 63,
         10,  0, 63, 11, 63, 12, 63, 13, 63, 14, 63, 15, 63, 16]),
 tensor([63,  2, 63,  3, 63,  4, 63,  5, 63,  6, 63,  7, 63,  8, 63,  9, 63, 10,
          0, 63, 11, 63, 12, 63, 13, 63, 14, 63, 15, 63, 16, 63]))

In [34]:
#model parameters count 
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

#model architecture 
print(u_model)

1217
UstaModel(
  (embedding): Embedding(65, 4)
  (pos_embedding): Embedding(32, 4)
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): Linear(in_features=4, out_features=4, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=4, out_features=4, bias=True)
        (up_proj): Linear(in_features=4, out_features=4, bias=True)
        (down_proj): Linear(in_features=4, out_features=4, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projectio

In [35]:
u_model.embedding.weight.shape

torch.Size([65, 4])

In [36]:
out0 = u_model(dataset.inputs[0])
out0.shape

torch.Size([32, 65])

In [37]:
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss()

In [38]:
loss = loss_fn(out0,dataset.targets[0])
loss

tensor(4.4145, grad_fn=<NllLossBackward0>)

In [39]:
#optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)
optimizer = torch.optim.AdamW(u_model.parameters(),lr = 1e-3)

In [40]:
for input , target in dataset :
    print(input.shape , target.shape)
    break 

torch.Size([32]) torch.Size([32])


In [ ]:
epoch = 100000

for epoch in range(epoch):
    total_loss = 0
    for input, target in dataset :
        pred = u_model(input)
        loss = loss_fn(pred,target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    average_loss = total_loss/ len(dataset)
    print(f"Epoch {epoch} loss : {loss.item()} average loss : {average_loss}")

In [44]:
new_tokens = tokens.detach().cpu().numpy().tolist()
new_tokens.pop()
new_tokens.pop()
new_tokens

[1, 63, 3, 63, 2, 63, 3, 63, 4, 63, 5, 63, 6, 63, 7, 63, 5, 63, 8, 63, 9]

In [49]:
import torch 
new_tokens = u_tokenizer.encode(prompt)
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(63)

out = u_model(torch.tensor(new_tokens))
probs = torch.softmax(out[-1] , dim =-1)
max_prob ,max_index = torch.max(probs,dim =-1)
max_prob , max_index , probs

(tensor(0.1332, grad_fn=<MaxBackward0>),
 tensor(0),
 tensor([1.3324e-01, 8.3396e-04, 6.8463e-04, 2.9764e-03, 1.5065e-03, 3.3795e-02,
         7.2093e-03, 1.0501e-02, 1.7151e-02, 1.8968e-02, 1.4568e-02, 2.0599e-02,
         2.3056e-02, 1.7206e-02, 1.7265e-02, 1.2096e-02, 3.2168e-02, 2.5097e-02,
         1.4431e-02, 3.7994e-03, 2.6504e-03, 2.1250e-03, 1.7954e-03, 1.7018e-02,
         1.6464e-03, 8.8753e-03, 2.9827e-02, 1.8959e-02, 2.3696e-02, 2.7958e-02,
         2.3520e-02, 4.2894e-03, 3.9584e-03, 6.4765e-03, 6.5486e-04, 3.0038e-03,
         3.5424e-03, 7.8352e-03, 5.4448e-03, 1.3592e-02, 2.0225e-02, 2.4457e-02,
         1.1664e-02, 1.2444e-02, 7.5185e-03, 4.2330e-03, 2.3032e-02, 1.7417e-02,
         3.4496e-02, 1.2528e-02, 2.2342e-02, 2.4068e-02, 2.4747e-02, 8.5468e-03,
         2.5074e-02, 9.6529e-03, 2.3430e-02, 7.0459e-03, 7.4280e-03, 2.4297e-02,
         1.1617e-02, 8.0753e-03, 1.9826e-02, 1.8152e-03, 8.2919e-09],
        grad_fn=<SoftmaxBackward0>))

In [53]:
#save model 
torch.save(u_model.state_dict() ,"u_model.pth")

#load model
u_model.load_state_dict(torch.load("u_model.pth"))

#generate text
new_tokens = u_tokenizer.encode(prompt)
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(63)
len(new_tokens)


24

In [54]:
out = u_model(torch.tensor(new_tokens))

porbs = torch.softmax(out[-1],dim =-1)
max_prob , max_index = torch.max(probs,dim=-1)
max_prob , max_index , probs

(tensor(0.1332, grad_fn=<MaxBackward0>),
 tensor(0),
 tensor([1.3324e-01, 8.3396e-04, 6.8463e-04, 2.9764e-03, 1.5065e-03, 3.3795e-02,
         7.2093e-03, 1.0501e-02, 1.7151e-02, 1.8968e-02, 1.4568e-02, 2.0599e-02,
         2.3056e-02, 1.7206e-02, 1.7265e-02, 1.2096e-02, 3.2168e-02, 2.5097e-02,
         1.4431e-02, 3.7994e-03, 2.6504e-03, 2.1250e-03, 1.7954e-03, 1.7018e-02,
         1.6464e-03, 8.8753e-03, 2.9827e-02, 1.8959e-02, 2.3696e-02, 2.7958e-02,
         2.3520e-02, 4.2894e-03, 3.9584e-03, 6.4765e-03, 6.5486e-04, 3.0038e-03,
         3.5424e-03, 7.8352e-03, 5.4448e-03, 1.3592e-02, 2.0225e-02, 2.4457e-02,
         1.1664e-02, 1.2444e-02, 7.5185e-03, 4.2330e-03, 2.3032e-02, 1.7417e-02,
         3.4496e-02, 1.2528e-02, 2.2342e-02, 2.4068e-02, 2.4747e-02, 8.5468e-03,
         2.5074e-02, 9.6529e-03, 2.3430e-02, 7.0459e-03, 7.4280e-03, 2.4297e-02,
         1.1617e-02, 8.0753e-03, 1.9826e-02, 1.8152e-03, 8.2919e-09],
        grad_fn=<SoftmaxBackward0>))

In [60]:
loaded_model = UstaModel(65 , embedding_dim=4 , num_heads = 2, context_length = 32 ,num_layers = 3)
loaded_model.load_state_dict(torch.load("u_model.pth"))
loaded_model

UstaModel(
  (embedding): Embedding(65, 4)
  (pos_embedding): Embedding(32, 4)
  (layers): Sequential(
    (0): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): Linear(in_features=4, out_features=4, bias=True)
      )
      (norm1): UstaLayerNorm()
      (mlp): UstaMLP(
        (gate_proj): Linear(in_features=4, out_features=4, bias=True)
        (up_proj): Linear(in_features=4, out_features=4, bias=True)
        (down_proj): Linear(in_features=4, out_features=4, bias=True)
        (gelu): GELU()
      )
      (norm2): UstaLayerNorm()
    )
    (1): UstaDecoderBlock(
      (self_attention): UstaMultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): L